# 5. Fusion RAG

**Fusion RAG** (also called **RAG-Fusion**) improves search by asking the **same question in many different ways**, then **combining** all the results.

This helps find documents that a single search might miss.

## Why do we need it?

One question can be asked in many ways. For example, "How to lose weight?" could also be:
- "Tips to reduce body fat"
- "Ways to become slim"
- "How to burn calories"

Each version may find **different useful documents**. Fusion RAG uses **all of them** and merges the best results.

## How Fusion RAG works

```
Original question
        |
        v
[1] Make several versions of the question (using AI)
        |
        v
[2] Search with EACH version separately
        |
        v
[3] Combine all results and re-rank them
        |
        v
[4] Use the top results to generate the answer
```

## The key idea: Reciprocal Rank Fusion (RRF)

To merge results from different searches fairly, Fusion RAG uses a simple scoring trick called **Reciprocal Rank Fusion (RRF)**.

The rule is simple:
- A document that appears **near the top** in many searches gets a **higher score**.
- Score for each appearance = `1 / (rank + k)` (k is a small constant like 60).
- Add up the scores from all searches.

## A simple example of combining results

Below we have results from 2 different searches. We use RRF to merge and re-rank them.

In [ ]:
# Results from two different question versions
# (each list is ordered: best match first)
search_1 = ["docA", "docB", "docC"]
search_2 = ["docB", "docA", "docD"]

def reciprocal_rank_fusion(searches, k=60):
    scores = {}
    for result_list in searches:
        for rank, doc in enumerate(result_list):
            # rank starts at 0, so we use rank+1
            scores[doc] = scores.get(doc, 0) + 1 / (rank + 1 + k)
    # sort documents by total score (highest first)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

final = reciprocal_rank_fusion([search_1, search_2])

print("Final combined ranking:")
for doc, score in final:
    print(doc, "-> score:", round(score, 4))

## What the result shows

Documents that appeared in **both** searches (like `docA` and `docB`) get **higher scores** and move to the **top**.

This way, the most agreed-upon documents are used to answer the question.

## Key points to remember

- Fusion RAG asks the question in **many different ways**.
- It **searches with each version** separately.
- It **combines** all results using **Reciprocal Rank Fusion (RRF)**.
- Documents found by **many searches** rank higher.
- This finds better documents that a single search might miss.